# Predicción horaria con VS-GRU

VS-GRU procesa cada variable temporal por separado, con un estado recurrente por variable. La celda imputa los faltantes por decaimiento hacia la media de entrenamiento y usa la proporción de faltantes dentro de las horas válidas de cada ventana. El clasificador combina los estados finales con el contexto del paciente.

Antes de empezar, ejecuta `uv sync --extra model --group dev` en la raíz del proyecto, selecciona el kernel de `.venv` y comprueba que `cleaning.ipynb` haya exportado las particiones.

In [1]:
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
from tensorflow import keras

from appr_project.data import (
    PatientHours,
    Standardization,
    WindowBatches,
    load_partition,
    split_patients,
)
from appr_project.evaluation import predict_hours, summarize_metrics
from appr_project.training import Hyperparameters, train_experiment

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
DATA_DIR = ROOT / "data" / "processed"

## 1. Configuración del piloto

El piloto usa hasta 2.000 pacientes de A y una sola partición de validación. Los dos ensayos comparten partición e hiperparámetros y comparan una salida directa con una capa oculta de 32 neuronas y ReLU. El modelo se selecciona por precisión promedio (AP) en validación.

Para medir el aporte de B con la arquitectura fija, cambia `source` a `"A+B"`, pon `trials=1` y copia el valor ganador de `use_hidden_dense` a `base`. Usa la misma configuración de muestreo y partición que en los notebooks semánticos para comparar arquitecturas dentro de cada origen.

In [4]:
settings = {
    "architecture": "vs_gru",
    "source": "A",
    "sample_patients": 2000,
    "validation_fraction": 0.20,
    "folds": None,
    "search": "tpe",
    "trials": 20,
    "epochs": 30,
    "patience": 4,
    "seed": 2026,
}

base = Hyperparameters(
    window=12,
    learning_rate=0.001,
    batch_size=128,
    use_hidden_dense=False,
)

## 2. Inspeccionar la partición y un batch

Se muestran los pacientes, sus horas y cuántos tienen alguna etiqueta positiva. En CV, la vista previa corresponde al primer fold. La muestra y la partición se reproducen al entrenar con la misma semilla.

Cada entrada temporal tiene dimensiones `[batch, horas, variables, 4]`. Los cuatro canales son el valor estandarizado, la máscara de medición, el tiempo desde la medición previa y su valor. La máscara `valid_hours` marca las horas reales de la ventana.


In [5]:
frame = load_partition(DATA_DIR, "train", settings["source"])

train_ids, val_ids = next(
    split_patients(
        frame,
        validation_fraction=settings["validation_fraction"],
        folds=settings["folds"],
        seed=settings["seed"],
        sample_patients=settings["sample_patients"],
    )
)

train_preview = frame.loc[frame.patient_id.isin(train_ids)]
val_preview = frame.loc[frame.patient_id.isin(val_ids)]

summary = []
for name, partition in (("train", train_preview), ("validation", val_preview)):
    patient_labels = partition.groupby("patient_id").SepsisLabel.max()
    summary.append(
        {
            "partición": name,
            "pacientes": len(patient_labels),
            "pacientes positivos": int(patient_labels.sum()),
            "horas": len(partition),
        }
    )
display(pd.DataFrame(summary).set_index("partición"))

,pacientes,pacientes positivos,horas
partición,,,
train,1600,141,62270
validation,400,35,15554


In [6]:
scaler = Standardization.fit(train_preview)
first_patient = train_preview.loc[train_preview.patient_id.eq(train_ids[0])]
hours = PatientHours(first_patient, scaler)
batches = WindowBatches(hours, base.window, base.batch_size)
inputs, labels = batches[0]

print("Formas de entrada:", {name: array.shape for name, array in inputs.items()})
print("Forma de las etiquetas:", labels.shape)
print("Paciente:", hours.metadata["patient_id"].iloc[0])
print(
    "Horas válidas de la primera ventana:",
    inputs["valid_hours"][0].astype(int).tolist(),
)
print("Etiqueta de la primera ventana:", int(labels[0, 0]))

print("Variables temporales:", scaler.temporal)

Formas de entrada: {'temporal': (11, 12, 26, 4), 'valid_hours': (11, 12), 'static': (11, 10)}
Forma de las etiquetas: (11, 1)
Paciente: A_015805
Horas válidas de la primera ventana: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
Etiqueta de la primera ventana: 0
Variables temporales: ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Creatinine', 'Glucose', 'Lactate', 'Magnesium', 'Phosphate', 'Potassium', 'Bilirubin_total', 'Hct', 'Hgb', 'WBC', 'Platelets']


## 3. Entrenar y revisar los resultados

Cada ejecución guarda el modelo, la configuración, las particiones, las métricas y los tiempos en `runs/`. La columna `use_hidden_dense` identifica la variante de cada ensayo.

Para comparar contra `modeling_semantic.ipynb` y `modeling_semantic_3.ipynb`, comprueba que coincidan las asignaciones de pacientes en `splits.csv` y usa el mismo origen de prueba.

In [7]:
run_dir = (
    ROOT
    / "runs"
    / f"notebook-vs-gru-{settings['source']}-{datetime.now():%Y%m%d-%H%M%S-%f}"
)
validation_metrics = train_experiment(frame, run_dir, base=base, **settings)

print(f"Resultados: {run_dir}")
display(validation_metrics.round(4))

c:\Users\palom\Documents\College\APPR\APPR_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-11-16 12:31:23,120] A new study created in memory with name: no-name-fbbc6686-4e08-4725-89a6-350bc4cef480


Trial 1 (1/20), fold 1/1: Hyperparameters(window=12, units=16, dense_units=64, dropout=0.2, learning_rate=0.001, batch_size=128, use_hidden_dense=False)

Epoch 1/30
487/487 - 20s - 40ms/step - loss: 0.3669 - pr_auc: 0.0230 - val_loss: 0.1032 - val_pr_auc: 0.0362
Epoch 2/30
487/487 - 13s - 27ms/step - loss: 0.1033 - pr_auc: 0.0338 - val_loss: 0.1007 - val_pr_auc: 0.0611
Epoch 3/30
487/487 - 13s - 26ms/step - loss: 0.1001 - pr_auc: 0.0484 - val_loss: 0.0991 - val_pr_auc: 0.0869
Epoch 4/30
487/487 - 12s - 25ms/step - loss: 0.0979 - pr_auc: 0.0645 - val_loss: 0.0980 - val_pr_auc: 0.0959
Epoch 5/30
487/487 - 13s - 26ms/step - loss: 0.0961 - pr_auc: 0.0723 - val_loss: 0.0973 - val_pr_auc: 0.1016
Epoch 6/30
487/487 - 20s - 40ms/step - loss: 0.0946 - pr_auc: 0.0827 - val_loss: 0.0969 - val_pr_auc: 0.1038
Epoch 7/30
487/487 - 14s - 29ms/step - loss: 0.0934 - pr_auc: 0.0898 - val_loss: 0.0965 - val_pr_auc: 0.1051
Epoch 8/30
487/487 - 13s - 26ms/step - loss: 0.0920 - pr_auc: 0.1020 - val_loss: 0.

,source,patients,hours,positive_hours,positive_hour_fraction,average_precision,auroc,utility,alert_fraction,recall_at_threshold,threshold
0,all,400,15554,338,0.0217,0.1464,0.7798,0.3719,0.2346,0.6538,0.0301
1,A,400,15554,338,0.0217,0.1464,0.7798,0.3719,0.2346,0.6538,0.0301


In [8]:
trial_results = pd.read_csv(run_dir / "trials.csv")
display(
    trial_results[
        ["trial", "use_hidden_dense", "mean_validation_ap", "std_validation_ap"]
    ].round(4)
)
config = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
print("Capa oculta seleccionada:", config["hyperparameters"]["use_hidden_dense"])

fold_metrics = pd.read_csv(run_dir / "fold_metrics.csv")
timing_columns = [
    "trial",
    "fold",
    "train_windows",
    "validation_windows",
    "preparation_seconds",
    "epochs_run",
    "seconds_per_epoch",
]
display(fold_metrics.loc[fold_metrics.source.eq("all"), timing_columns].round(2))


run_timing = json.loads((run_dir / "run_timing.json").read_text(encoding="utf-8"))
print(f"Tiempo total de ejecución: {run_timing['total_seconds']:.2f} s")

,trial,use_hidden_dense,mean_validation_ap,std_validation_ap
0,1,False,0.1196,0.0
1,2,True,0.1085,0.0
2,3,True,0.0995,0.0
3,4,True,0.1231,0.0
4,5,False,0.0981,0.0
5,6,True,NaN,NaN
6,7,False,NaN,NaN
7,8,False,0.1217,0.0
8,9,True,NaN,NaN
9,10,False,NaN,NaN


Capa oculta seleccionada: True


,trial,fold,train_windows,validation_windows,preparation_seconds,epochs_run,seconds_per_epoch
0,1,1,62270,15554,1.33,22,13.90
2,2,1,62270,15554,0.71,7,19.92
4,3,1,62270,15554,0.87,30,16.70
6,4,1,62270,15554,1.49,10,15.91
8,5,1,62270,15554,0.55,14,13.88
10,8,1,62270,15554,1.08,10,30.12
12,11,1,62270,15554,0.61,7,25.79
14,12,1,62270,15554,1.18,10,45.88
16,14,1,62270,15554,1.12,7,24.00
18,15,1,62270,15554,1.46,7,35.28


Tiempo total de ejecución: 5754.89 s


## 4. Evaluación final opcional

Activa esta celda cuando hayas elegido la configuración definitiva. La utilidad y la proporción de horas con alerta en test se calculan con el umbral guardado en la corrida. Los resultados se muestran y se guardan por origen y de forma agregada. Usa `positive_hour_fraction` como referencia para el AP.


In [ ]:
evaluate_test = False
test_source = "A+B"  # También admite "A" o "B".

if evaluate_test:
    config = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
    saved_model = keras.models.load_model(run_dir / "best.keras", compile=False)
    saved_scaler = Standardization(**config["standardization"])
    test = load_partition(DATA_DIR, "test", test_source)
    test_batches = WindowBatches(
        PatientHours(test, saved_scaler),
        config["hyperparameters"]["window"],
        config["hyperparameters"].get("batch_size", config.get("batch_size")),
    )
    test_predictions = predict_hours(saved_model, test_batches)
    test_metrics = summarize_metrics(test_predictions, config["threshold"])
    test_predictions.to_csv(
        run_dir / f"test_{test_source}_predictions.csv", index=False
    )
    test_metrics.to_csv(run_dir / f"test_{test_source}_metrics.csv", index=False)
    display(test_metrics.round(4))